# 07 - Exploratory Data Analysis

Inspect the beat table before any modelling: its shape, types, missing values, class balance and outliers, and how the features behave. **No model is trained here.** Every plot is there to answer one stated question, and what we learn here decides what the split, the models and the evaluation need to watch out for.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from src.feature_extraction import FEATURE_COLUMNS, get_beat_waveform, load_beat_table

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. One clean DataFrame: features + labels + metadata

The features and labels are already combined - `build_beat_table` in `src/feature_extraction.py` writes one row per heartbeat holding the metadata (`record`, `r_peak_sample`, `symbol`, `label`, ...), the label, and the 13 features. This section loads that table with explicit types, enforces a clean schema and checks it, so later notebooks can rely on it.

In [ ]:
raw = load_beat_table()
print("shape:", raw.shape, "(rows = heartbeats, columns = metadata + label + features)")
raw.head(3)

In [ ]:
META_COLUMNS = [c for c in raw.columns if c not in FEATURE_COLUMNS]

df = raw[META_COLUMNS + FEATURE_COLUMNS].copy()
df["label"] = pd.Categorical(df["label"], categories=["Normal", "Abnormal"], ordered=True)
df["record"] = df["record"].astype("category")
df["symbol"] = df["symbol"].astype("category")

# structural checks: fail loudly if the table is not what later stages assume
assert not df.duplicated(["record", "r_peak_sample"]).any(), "a beat appears twice"
assert (df.is_abnormal == (df.label == "Abnormal").astype(int)).all(), "is_abnormal disagrees with label"
assert np.isfinite(df[FEATURE_COLUMNS].to_numpy()).all(), "NaN or inf in the features"
assert df.groupby("record", observed=True).time_s.apply(lambda s: s.is_monotonic_increasing).all(), "beats out of order"

print("Columns by role")
print("  metadata / target:", META_COLUMNS)
print("  features         :", FEATURE_COLUMNS)
print()
print(pd.DataFrame({"raw dtype": raw.dtypes.astype(str), "clean dtype": df[raw.columns].dtypes.astype(str)}).to_string())
print()
print(f"Memory: {raw.memory_usage(deep=True).sum() / 1e6:.1f} MB raw -> {df.memory_usage(deep=True).sum() / 1e6:.1f} MB clean")

`record`, `symbol` and `label` are categorical rather than free text (smaller in memory, and `label` has an explicit order). `symbol` is kept for error analysis only - it is the raw annotation, so it must **never** be used as a feature (it would leak the answer). The checks above passed: no beat is duplicated, `is_abnormal` agrees with `label`, every feature is finite, and beats are in time order within each record.

The three objects the modelling stage will use:

In [ ]:
X = df[FEATURE_COLUMNS]     # inputs
y = df["is_abnormal"]       # target: 1 = Abnormal
groups = df["record"]       # which recording each beat came from (for the record-level split)
print("X:", X.shape, "| y:", y.shape, "| records:", groups.nunique())

## 2. Missing values

*Question: is anything missing or non-finite?*

In [ ]:
missing = df.isna().sum()
print("Total missing values:", int(missing.sum()))
missing[missing > 0]

No values are missing, and that is by construction rather than luck: the only beats that would have had NaN features (the first and last beat of each record, which have no previous or next R peak for the RR features) were dropped when the table was built - at most 2 per record, counted in notebook 06. Nothing is imputed or filled in.

## 3. Class distribution

*Questions: how imbalanced is the target overall, and how unevenly is it spread across records? (The second matters because we split by record.)*

In [ ]:
label_counts = df.label.value_counts()
print(label_counts.to_string())
print(f"Abnormal: {100 * df.is_abnormal.mean():.1f}% of beats  |  imbalance {label_counts['Normal'] / label_counts['Abnormal']:.1f} : 1")

per_record = df.groupby("record", observed=True).agg(beats=("label", "size"), abnormal=("is_abnormal", "sum"))
per_record["normal"] = per_record.beats - per_record.abnormal
per_record["abnormal_pct"] = (100 * per_record.abnormal / per_record.beats).round(1)
per_record["share_of_all_abnormal_pct"] = (100 * per_record.abnormal / per_record.abnormal.sum()).round(1)
per_record.sort_values("abnormal_pct", ascending=False)

In [ ]:
order = per_record.sort_values("abnormal_pct").index
sym = (df.groupby("symbol", observed=True)
         .agg(n=("symbol", "size"), label=("label", "first"))
         .sort_values("n"))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.8), gridspec_kw={"width_ratios": [1.3, 1]})

ax = axes[0]
ax.barh(order.astype(str), per_record.loc[order, "normal"], color="tab:blue", label="Normal")
ax.barh(order.astype(str), per_record.loc[order, "abnormal"], left=per_record.loc[order, "normal"],
        color="tab:red", label="Abnormal")
for i, rec in enumerate(order):
    ax.text(per_record.loc[rec, "beats"] + 25, i, f"{per_record.loc[rec, 'abnormal_pct']:.0f}%", va="center", fontsize=8)
ax.set_title("Beats per record (text = % abnormal)")
ax.set_xlabel("Number of beats")
ax.set_ylabel("Record")
ax.legend(loc="lower right")

ax = axes[1]
ax.barh(sym.index.astype(str), sym.n, color=["tab:red" if l == "Abnormal" else "tab:blue" for l in sym.label])
ax.set_xscale("log")
for i, n in enumerate(sym.n):
    ax.text(n * 1.1, i, str(n), va="center", fontsize=8)
ax.set_title("Beats per annotation symbol (log scale; red = Abnormal)")
ax.set_xlabel("Number of beats (log)")
ax.set_ylabel("MIT-BIH symbol")

fig.tight_layout()
fig.savefig("../results/figures/12_class_distribution.png", dpi=120)
plt.show()

- **Overall imbalance is moderate: 6.4 : 1** (13.6% abnormal). 5,555 abnormal beats is enough for classical models, but a model that always answers "Normal" would already score 86.4% accuracy, so accuracy alone will be meaningless.
- **It is very unevenly spread across records.** The abnormal share runs from 1.8% (record 234) to 77.8% (record 232). Three records (232, 200, 233) hold 55% of all abnormal beats, record 232 alone holds 24.5%, and 10 of the 17 records are under 5% abnormal.
- **What that means for the split:** a random assignment of whole records to train/test could put 232 (and a large part of the abnormal data) entirely on one side, so the test score would mostly describe how the model treats one patient. The record-level split has to be balanced on abnormal load, and results should be reported per record, not just as one pooled number.
- **Both classes are mixtures.** Abnormal is mostly `V` (3,012) and `A` (2,057), with `F` 389, `J` 81 and `a` 16; Normal is `N` (29,318), `R` (4,076), `L` (1,985) and 6 `j`. The rare symbols are far too small to learn separately - which is why binary was the right scope.

## 4. Feature summary and obvious outliers

*Questions: what do the feature values look like, are any physically impossible, and are the extreme values junk or real?*

In [ ]:
summary = df[FEATURE_COLUMNS].describe().T
summary["skew"] = df[FEATURE_COLUMNS].skew()
summary.round(3)

In [ ]:
# "extreme" = beyond 3 x IQR outside the quartiles (a deliberately generous fence)
q1, q3 = X.quantile(0.25), X.quantile(0.75)
iqr = q3 - q1
extreme = (X < q1 - 3 * iqr) | (X > q3 + 3 * iqr)

outlier_table = pd.DataFrame({
    "n_extreme": extreme.sum(),
    "pct_of_beats": (100 * extreme.mean()).round(2),
    "pct_of_those_abnormal": [round(100 * df.loc[extreme[f], "is_abnormal"].mean(), 1) if extreme[f].any() else np.nan
                              for f in FEATURE_COLUMNS],
}).sort_values("pct_of_beats", ascending=False)
print(f"Baseline for comparison: {100 * df.is_abnormal.mean():.1f}% of all beats are abnormal")
outlier_table

In [ ]:
# limits implied by how the table was built - a value outside them would point to a bug
FS = 360  # MIT-BIH sampling rate
print(f"rr_pre_s range: {df.rr_pre_s.min():.3f} - {df.rr_pre_s.max():.1f} s")
close = df[df.rr_pre_s < 0.300]
print(f"  beats closer than 0.300 s to the previous beat: {len(close)}, symbols {close.symbol.value_counts().loc[lambda s: s > 0].to_dict()}")
print(f"heart_rate_bpm max: {df.heart_rate_bpm.max():.0f}")
print("  (the 300 ms / 200 bpm spacing is enforced on the detector's candidates; the +-50 ms search-back")
print("   then moves each peak, so slightly closer spacing is possible - these are not bugs)")
window_ms = (2 * round(0.1 * FS) + 1) / FS * 1000
print(f"qrs_fwhm_ms max: {df.qrs_fwhm_ms.max():.1f} ms vs the {window_ms:.0f} ms QRS window -> the half-height search is never truncated")
print(f"ann_offset_ms max: {df.ann_offset_ms.max():.1f} ms (label matching tolerance is 150 ms)")

Do the extreme values look like recording junk or like real beats? Look at them rather than assume: the four **Normal**-labeled beats with the highest energy (a normal beat is normally narrow and low-energy, so these are the most suspicious), and the four beats with the longest gap since the previous beat (a long gap can mean the detector missed a beat).

In [ ]:
normal_high_energy = df[df.label == "Normal"].nlargest(4, "energy_mv2s")
longest_gap = df.nlargest(4, "rr_pre_s")

fig, axes = plt.subplots(2, 4, figsize=(17, 7), sharex=True)
for ax, (_, row) in zip(axes[0], normal_high_energy.iterrows()):
    t, w = get_beat_waveform(str(row.record), int(row.r_peak_sample))
    ax.plot(t, w, color="black", linewidth=1.1)
    ax.axvline(0, color="red", linestyle="--", linewidth=0.8)
    ax.set_title(f"rec {row.record} symbol {row.symbol}  energy={row.energy_mv2s:.2f}", fontsize=9)
for ax, (_, row) in zip(axes[1], longest_gap.iterrows()):
    t, w = get_beat_waveform(str(row.record), int(row.r_peak_sample))
    ax.plot(t, w, color="black", linewidth=1.1)
    ax.axvline(0, color="red", linestyle="--", linewidth=0.8)
    ax.set_title(f"rec {row.record} symbol {row.symbol}  rr_pre={row.rr_pre_s:.2f}s", fontsize=9)
axes[0][0].set_ylabel("Highest-energy Normal beats\nAmplitude (mV)")
axes[1][0].set_ylabel("Longest gap before the beat\nAmplitude (mV)")
for ax in axes[1]:
    ax.set_xlabel("Time relative to R peak (ms)")
fig.tight_layout()
fig.savefig("../results/figures/13_outlier_beats.png", dpi=120)
plt.show()

**Two different kinds of "extreme" - one to keep, one to be wary of.**

- *Extremes that are the signal.* For `dominant_deflection_mv`, `qrs_fwhm_ms`, `energy_mv2s` and `amp_min`, between 86% and 99.9% of the extreme values are Abnormal beats (baseline: 13.6%). These are wide, large or inverted ectopic beats - exactly what the model must find. **Removing or clipping them would delete the class we are trying to detect.** They call for robust handling (scale-insensitive models, transforms), not dropping rows.
- *Extremes that are recording artifacts.* The four highest-energy **Normal**-labeled beats all come from record 116 and contain abrupt baseline steps and swings that the 0.5 Hz filter did not remove (the top-left beat drops from +1.3 mV to -1.7 mV within about 50 ms). There are few of them, but they are real noise inside the Normal class.
- *RR outliers.* `rr_ratio` shows 21% "extreme", but that is the 3 x IQR fence being far too tight for a feature whose core is 0.97-1.04, not 21% junk: ordinary rhythm variability and premature beats fall outside it. The genuinely extreme RR values are the long gaps, examined next.
- The five beats closer than 0.3 s to their predecessor are all `V` (records 233 and 205) - real close-coupled premature beats, not bugs.

### Are the long gaps real pauses, or beats the detector missed?

`rr_pre_s` reaches 11.5 s. The annotations can tell us which it is: count the annotated beats that lie *inside* each gap (ignoring 150 ms at each end, where the two detected beats themselves sit). Zero means a genuine pause; one or more means our detector missed them.

In [ ]:
from src.feature_extraction import BEAT_SYMBOLS
from src.preprocessing import load_record


def annotated_beat_samples(record_name):
    _, annotation = load_record(record_name)
    keep = np.isin(np.asarray(annotation.symbol), list(BEAT_SYMBOLS))
    return np.asarray(annotation.sample)[keep]


long_gap = df[df.rr_pre_s > 2.0].copy()
beat_samples = {str(r): annotated_beat_samples(str(r)) for r in long_gap.record.unique()}
margin = int(0.15 * FS)

hidden = []
for _, row in long_gap.iterrows():
    samples = beat_samples[str(row.record)]
    gap_start = row.r_peak_sample - int(round(row.rr_pre_s * FS))
    hidden.append(int(((samples > gap_start + margin) & (samples < row.r_peak_sample - margin)).sum()))
long_gap["annotated_beats_inside_gap"] = hidden

missed = long_gap.annotated_beats_inside_gap >= 1
print(f"Beats preceded by a gap > 2 s: {len(long_gap)}   (by record: {long_gap.record.value_counts().head(3).to_dict()} ...)")
print(f"  genuine pauses (no annotated beat inside the gap): {int((~missed).sum())}")
print(f"  detector missed beats inside the gap:              {int(missed.sum())}  ({100 * missed.sum() / len(df):.2f}% of all beats)")
long_gap.sort_values("rr_pre_s", ascending=False)[["record", "symbol", "rr_pre_s", "annotated_beats_inside_gap"]].head(6)

Of the 172 beats preceded by a gap longer than 2 s, 119 are genuine pauses (no annotated beat inside; 128 of the 172 sit in record 232, which really does have long pauses). But **53 are detector failures**: the gap hides between 1 and 16 annotated beats that we never detected - including all four of the longest gaps in the gallery above (10-16 missed beats each). For those beats `rr_pre_s` and `rr_ratio` describe a hole in our detection, not the patient's rhythm.

That is 0.13% of the table, so no headline number will move, but two consequences follow. First, the RR features are only as trustworthy as the detector - one more reason to report per-record results. Second, heavy-tailed RR values (`rr_ratio` reaches 16) would distort a linear model, so at modelling time the RR features should be clipped to physiological bounds or log-transformed. A fixed rule of that kind is not fitted on the data, so it cannot leak test information.

## 5. Do the features differ between the classes?

*Question: for each feature, do Normal and Abnormal beats have visibly different distributions?* Densities (so the 13.6% minority is comparable in height); the x-axis is limited to the 0.5th-99.5th percentile so a few extremes do not squash the picture.

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(15, 17))
for ax, f in zip(axes.ravel(), FEATURE_COLUMNS):
    lo, hi = df[f].quantile([0.005, 0.995])
    bins = np.linspace(lo, hi, 60)
    for label, color in [("Normal", "tab:blue"), ("Abnormal", "tab:red")]:
        values = df.loc[df.label == label, f]
        ax.hist(values[(values >= lo) & (values <= hi)], bins=bins, density=True, alpha=0.5, color=color, label=label)
    ax.set_title(f)
for ax in axes.ravel()[len(FEATURE_COLUMNS):]:
    ax.axis("off")
axes[0][0].legend()
fig.tight_layout()
fig.savefig("../results/figures/14_feature_distributions.png", dpi=110)
plt.show()

What the histograms show (densities, so the class sizes do not hide anything):

- **Timing separates well.** Abnormal beats pile up at short `rr_pre_s` (about 0.4-0.55 s), and `rr_ratio` has a distinct second mode at 0.4-0.7 that Normal beats almost never reach. Both classes share the tall peak at a ratio of 1.0 (regular rhythm), so a ratio near 1 does not rule out abnormal.
- **`qrs_fwhm_ms`:** Normal beats sit mostly below about 40 ms; abnormal beats have a long tail out to 80 ms. The comb-like steps are real - at 360 Hz one sample is 2.8 ms, so the width can only take multiples of that.
- **Inverted beats:** `dominant_deflection_mv` and `amp_min` show a separate abnormal cluster near -2 mV that Normal beats simply do not have - a clean cue for the subset of ectopic beats that are inverted.
- **Amplitude features overlap heavily**, and their abnormal peaks (`r_amplitude_mv` and `amp_max` near 0.7-0.9 mV, `amp_std` near 0.17) look like the low-gain record 232 (78% abnormal, lowest amplitude of all) rather than beat physiology. That is checked next with the within-record AUC.

### How well does each feature separate the classes on its own?

A number to go with the picture. For each feature, the AUC is the probability that a randomly chosen Abnormal beat has a *higher* value than a randomly chosen Normal beat (0.5 = no separation; near 0 or 1 = strong; above 0.5 means abnormal beats tend to be higher). This is a rank statistic - **no model is fitted**.

It is computed two ways because they answer different questions:
- **Pooled** - all beats together. Confounded by *which record* a beat is from: record 232 is 78% abnormal, so any feature that just tells 232 apart from the rest will look useful.
- **Within-record** - the AUC inside each record separately, then averaged. Patient identity cannot help here, so this is the honest measure of whether a feature carries information about the beat itself. The last column counts the records whose direction agrees with the average.

In [ ]:
pooled, within, agree = {}, {}, {}
for f in FEATURE_COLUMNS:
    pooled[f] = roc_auc_score(df.is_abnormal, df[f])
    per_rec = [roc_auc_score(g.is_abnormal, g[f]) for _, g in df.groupby("record", observed=True)
               if 0 < g.is_abnormal.sum() < len(g)]
    within[f] = float(np.mean(per_rec))
    n_agree = sum(a > 0.5 for a in per_rec) if within[f] > 0.5 else sum(a < 0.5 for a in per_rec)
    agree[f] = f"{n_agree}/{len(per_rec)}"

separation = pd.DataFrame({"pooled_auc": pooled, "within_record_auc": within, "records_agreeing": agree})
separation["strength"] = (separation.within_record_auc - 0.5).abs()
separation.sort_values("strength", ascending=False).drop(columns="strength").round(3)

Reading the two AUC columns together (0.5 = no separation; the distance from 0.5 is what counts, the direction only says which class is higher):

- **Timing is the strongest and most consistent group.** `rr_pre_s` has a within-record AUC of 0.074 in **17 of 17** records - abnormal beats always arrive earlier. `heart_rate_bpm` is the same information mirrored (0.926). `rr_ratio` follows (0.171, 16/17). The pooled AUC (0.188) is weaker than the within-record one because patients differ in resting heart rate, which blurs an absolute interval.
- **Width and energy come next:** `qrs_fwhm_ms` 0.766, `amp_std` 0.749, `energy_mv2s` 0.747 (direction agrees in 12-13 of 17 records).
- **The amplitude features are a trap.** `r_amplitude_mv`, `dominant_deflection_mv` and `amp_max` have pooled AUCs of 0.32-0.35, so they *look* like they separate the classes. Within a record the AUCs *average* 0.46-0.47 - but only 11 of 17 records agree on the direction, so that average is hiding effects of opposite sign: the feature is informative inside some patients and inverted in others (notebook 08 shows this on the development patients alone, where `r_amplitude_mv` reaches 0.29). As a single rule a model could learn, it is close to useless, and the pooled signal is largely patient identity: record 232 has the lowest amplitude and 78% abnormal beats. A model trained on these would partly learn "which patient", and that would not transfer. `amp_mean` carries nothing either way (0.49).
- `qrs_p2p_mv` shows the opposite pattern (pooled 0.54, within-record 0.69): patient scale hides a genuine signal.

## 6. Redundancy between features

*Question: which features carry nearly the same information?* Spearman correlation (rank-based, so outliers and monotonic-but-nonlinear relations such as `heart_rate_bpm = 60 / rr_pre_s` are handled fairly).

In [ ]:
corr = df[FEATURE_COLUMNS].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10.5, 9))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=60, ha="right")
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Spearman correlation between features")
fig.tight_layout()
fig.savefig("../results/figures/15_feature_correlation.png", dpi=120)
plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
print("Feature pairs with |correlation| >= 0.9:")
pairs[pairs.abs() >= 0.9].sort_values(key=abs, ascending=False).round(3)

Five pairs are essentially duplicates (|correlation| >= 0.9): `rr_pre_s` / `heart_rate_bpm` (exactly -1, as predicted), `amp_std` / `energy_mv2s` (1.00 - energy is a function of the window's mean and spread), and three amplitude features that all track the R-peak height - `amp_max`, `r_amplitude_mv`, `dominant_deflection_mv` (0.99). The 13 columns therefore carry roughly nine independent pieces of information: a timing block (`rr_pre_s`, `rr_post_s`, `rr_ratio`), a size/energy block (`amp_std`, `amp_max`, `qrs_p2p_mv`, ...), plus `amp_min`, `amp_mean` and `qrs_fwhm_ms` (the last moderately tied to size: 0.66 with `amp_std`).

Tree models tolerate duplicates; a logistic regression's coefficients become unstable and hard to interpret with them, so the duplicates should be dropped rather than all 13 fed in.

## 7. Do records differ from each other?

*Question: is the same beat type comparable across patients?* Because the split will be by record, a feature whose scale depends on the patient will behave differently on unseen records. Normal beats only, so beat type is held constant.

In [ ]:
normal_beats = df[df.label == "Normal"]
amp_order = normal_beats.groupby("record", observed=True).r_amplitude_mv.median().sort_values().index

fig, ax = plt.subplots(figsize=(12, 4.6))
ax.boxplot([normal_beats.loc[normal_beats.record == r, "r_amplitude_mv"] for r in amp_order],
           tick_labels=amp_order.astype(str), showfliers=False)
ax.set_title("R-peak amplitude of Normal beats, by record (outliers hidden)")
ax.set_xlabel("Record")
ax.set_ylabel("r_amplitude_mv")
fig.tight_layout()
fig.savefig("../results/figures/16_amplitude_by_record.png", dpi=120)
plt.show()

### Would a per-record scale fix the two patient-dependent features?

A prototype, not a model: divide `rr_pre_s` and `r_amplitude_mv` by *that record's own median* (computed from the record's signal alone, no labels, so nothing about the test set leaks). Compare the separation before and after. The second comparison isolates atrial (`A`) beats against normal beats, since those were the ones the absolute timing features could not separate. Only records with at least 10 positive beats enter the within-record average.

In [ ]:
proto = df[["record", "symbol", "is_abnormal", "rr_pre_s", "r_amplitude_mv"]].copy()
for col in ["rr_pre_s", "r_amplitude_mv"]:
    proto[col + "_rel"] = proto[col] / proto.groupby("record", observed=True)[col].transform("median")


def pooled_and_within(data, col, min_pos=10):
    pooled = roc_auc_score(data.is_abnormal, data[col])
    per_rec = [roc_auc_score(g.is_abnormal, g[col]) for _, g in data.groupby("record", observed=True)
               if g.is_abnormal.sum() >= min_pos and g.is_abnormal.sum() < len(g)]
    return round(pooled, 3), round(float(np.mean(per_rec)), 3), len(per_rec)


a_vs_n = proto[proto.symbol.isin(["A", "N"])].assign(is_abnormal=lambda d: (d.symbol == "A").astype(int))
rows = []
for name, data in {"all Abnormal vs Normal": proto, "A beats vs N beats": a_vs_n}.items():
    for col in ["rr_pre_s", "r_amplitude_mv"]:
        pooled_raw, within, n_rec = pooled_and_within(data, col)
        pooled_scaled, _, _ = pooled_and_within(data, col + "_rel")
        rows.append((name, col, pooled_raw, pooled_scaled, within, n_rec))
pd.DataFrame(rows, columns=["comparison", "feature", "pooled_auc_raw", "pooled_auc_per_record_scaled",
                            "within_record_auc", "records_used"])

**How to read this table.** The within-record AUC cannot change when a feature is divided by a per-record constant (the ranking inside one record is untouched), so it is the *pooled* AUC before and after that carries the information: how well one global cut-off works across patients - which is exactly what a model trained on some records and tested on others needs.

- `rr_pre_s`: dividing by the record's median moves the pooled AUC toward its within-record value (0.188 -> 0.154, against 0.074 within a record). For atrial beats the effect is large (0.427 -> 0.247). Within a record, atrial beats arrive clearly earlier than normal beats (within-record AUC 0.055), but that was buried by differences in resting rate between patients - exactly the suspicion from notebook 06.
- `r_amplitude_mv`: after scaling, the pooled AUC (0.460) lands on the within-record value (0.464). So the apparent pooled signal (0.333) was **patient scale**, and as a single cross-patient rule the scaled feature carries little class information (its within-record value is an average of effects that change sign between patients - see the previous section).
- **Caveat:** a record's median is only a neutral "typical beat" reference when most of its beats are Normal. Record 232 is 78% abnormal, so its median is itself abnormal-dominated and normalising 232 will be less clean than the others. In real-time use the median would also have to come from a trailing window rather than the whole record.

## What the EDA tells us

1. **The table is structurally clean:** 40,940 beats x 20 columns, no missing values, duplicates or non-finite values, 13.6% abnormal.
2. **Class balance varies enormously by record** (1.8% to 77.8% abnormal); record 232 alone holds 24.5% of all abnormal beats.
3. **The best single features are timing** (`rr_pre_s`, `rr_ratio`) **and beat width/energy.** The amplitude features mostly encode which patient a beat came from, and their relationship to the label changes sign between patients.
4. **Extreme values in width, energy and deflection are the abnormal beats themselves - keep them.** A few artifact beats (record 116) and 53 beats whose RR features reflect detector misses do exist.
5. **The 13 features hold about nine independent pieces of information.**

## Proposed changes before modelling (nothing has been changed yet)

- Drop the four duplicates: `heart_rate_bpm`, `energy_mv2s`, `r_amplitude_mv`, `dominant_deflection_mv`.
- Add per-record-relative versions of the patient-dependent features (starting with `rr_pre_s` divided by the record median, and the amplitude features we keep).
- Clip the RR features to fixed physiological bounds (or log-transform them) for the linear model.
- Split by record, balanced on abnormal load; report per-record results and show the effect of record 232 explicitly.
- Keep `symbol` and `ann_offset_ms` out of the features - they are annotation-derived.